In [ ]:
import tensorflow as tf
# pyrefly: ignore [missing-import]
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
# pyrefly: ignore [missing-import]
from tensorflow.keras.applications import MobileNetV2

# Define constants
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_DIR = "data/dataset/train"
VAL_DIR = "data/dataset/val"

# 1. Load datasets automatically from directories
train_dataset = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"  # Class 0 or 1
)

val_dataset = image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

# Optimize data loading performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

# 2. Build the model using Transfer Learning
# Pre-trained MobileNetV2 handles feature extraction (edges, shapes)
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base_model.trainable = False  # Freeze original weights

# Add custom classification layers on top
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)  # Prevent overfitting
outputs = Dense(1, activation="sigmoid")(x)  # Output between 0 (CXR) and 1 (Non-CXR)

model = Model(inputs=base_model.input, outputs=outputs)

# 3. Compile the model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 4. Train the model
print("Starting training...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5
)

# 5. Save the trained model
model.save("chest_xray_detector.h5")
print("Model saved successfully as chest_xray_detector.h5")


In [ ]:
# pyrefly: ignore [missing-import]
import numpy as np
from tensorflow.keras.models import load_model
# pyrefly: ignore [missing-import]
from tensorflow.keras.preprocessing import image

# Load your trained model
model = load_model("chest_xray_detector.h5")

def predict_if_cxr(img_path):
    # Load and resize image to match model input
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Create batch dimension
    
    # Run prediction
    prediction = model.predict(img_array)[0][0]
    
    # Extract class names based on folder setup (assuming cxr=0, non_cxr=1)
    if prediction < 0.5:
        confidence = (1 - prediction) * 100
        print(f"Result: This IS a Chest X-Ray ({confidence:.2f}% confidence)")
    else:
        confidence = prediction * 100
        print(f"Result: This NOT a Chest X-Ray ({confidence:.2f}% confidence)")

# Example usage:
predict_if_cxr("test_image.jpg")
